# Create Multiscale Zarr Pyramid for Web Visualization

This notebook builds a multiscale Zarr pyramid from the MODIS LST Icechunk store so the dataset can be visualized as a slippy web map using [zarr-layer](https://github.com/carbonplan/zarr-layer) without creating a separate visualization copy or running a tile server.

**Stack**
- [topozarr](https://github.com/carbonplan/topozarr): generates coarsened multi-resolution levels following the [GeoZarr multiscales spec](https://github.com/zarr-developers/geozarr-spec)
- [zarr-layer](https://github.com/carbonplan/zarr-layer): TypeScript library that fetches and renders Zarr as a custom MapLibre/Mapbox layer, reprojecting on the GPU on the fly

**Why multiscales?**  
The full store is 18 000 × 36 000 pixels. Without overview levels, zarr-layer would need to fetch the entire array at every zoom level. Multiscales let the viewer fetch only the appropriate coarsened level for the current viewport — qualitatively matching the performance of a traditional tile server.

**Projection note**  
The store is in EPSG:4326 (WGS84 geographic). zarr-layer reprojects on the GPU client-side, so we write the pyramid in native projection and preserve pixel fidelity.

---
**Dependencies**: `topozarr` and `xproj` must be available in the pixi environment. Run `pixi install` before launching this notebook.

## Imports

In [ ]:
from pathlib import Path
import sys
import xarray as xr
import numpy as np
import zarr
import icechunk
import rioxarray
import xproj
from topozarr import create_pyramid, ZarrLayerVarConfig
import dask
import adlfs


sys.path.insert(0, str(Path('..').resolve()))
from icechunk_github_actions_demo import Config

config = Config('config/config_with_secrets_v1.txt')

dask.config.set(scheduler='threads')
zarr.config.set({'async.concurrency': 128})

## 1. Open the Icechunk Store

In [2]:
storage = icechunk.azure_storage(
    account=config.AZURE_STORAGE_ACCOUNT,
    container=config.AZURE_CONTAINER,
    prefix=config.ICECHUNK_PREFIX,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all', mask_and_scale=True)
ds['avg_daytime_lst'] = ds['avg_daytime_lst']-273.15
ds['max_daytime_lst'] = ds['max_daytime_lst']-273.15
ds

<xarray.Dataset> Size: 31GB
Dimensions:          (year: 3, latitude: 18000, longitude: 36000)
Coordinates:
  * year             (year) int64 24B 2020 2021 2022
  * latitude         (latitude) float64 144kB 90.0 89.98 89.97 ... -89.99 -90.0
  * longitude        (longitude) float64 288kB -180.0 -180.0 ... 180.0 180.0
    spatial_ref      int64 8B ...
Data variables:
    max_daytime_lst  (year, latitude, longitude) float64 16GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    avg_daytime_lst  (year, latitude, longitude) float64 16GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
Attributes:
    title:    MODIS MOD11A2 Annual Daytime Land Surface Temperature
    source:   MODIS Terra MOD11A2 Version 6.1 via Microsoft Planetary Computer

## 2. Prepare Dataset

topozarr needs:
1. A CRS assigned via `xproj` (`.proj.assign_crs()`)
2. The `spatial_ref` scalar coordinate removed — topozarr manages CRS metadata internally and the scalar causes issues during coarsening

Data comes out of the store as `float64` (xarray's `mask_and_scale=True` applies `scale_factor=0.02` and replaces fill values with `NaN`, promoting `uint16` → `float64`). `create_pyramid(method='mean')` propagates NaNs correctly, so no extra masking is needed.

In [3]:
# Extract CRS WKT from rioxarray before dropping spatial_ref
crs = ds.rio.crs
print('Native CRS:', crs)
print()
print('WKT:')
print(crs.to_wkt())

Native CRS: EPSG:4326

WKT:
GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]


In [4]:
# Drop the CF-convention spatial_ref scalar — xproj will carry the CRS instead
ds_clean = ds.drop_vars('spatial_ref')

# Assign CRS via xproj so topozarr can find it
ds_crs = ds_clean.proj.assign_crs(spatial_ref_crs={'wkt': crs.to_wkt()})

ds_crs

<xarray.Dataset> Size: 31GB
Dimensions:          (year: 3, latitude: 18000, longitude: 36000)
Coordinates:
  * year             (year) int64 24B 2020 2021 2022
  * latitude         (latitude) float64 144kB 90.0 89.98 89.97 ... -89.99 -90.0
  * longitude        (longitude) float64 288kB -180.0 -180.0 ... 180.0 180.0
  * wkt              int64 8B 0
Data variables:
    max_daytime_lst  (year, latitude, longitude) float64 16GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    avg_daytime_lst  (year, latitude, longitude) float64 16GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
Indexes:
    wkt      CRSIndex (crs=EPSG:4326)
Attributes:
    title:    MODIS MOD11A2 Annual Daytime Land Surface Temperature
    source:   MODIS Terra MOD11A2 Version 6.1 via Microsoft Planetary Computer

## 3. Create Multiscale Pyramid

### Choosing the number of levels

Each level coarsens by 2× in both spatial dimensions. The coarsest level (level 0) should be small enough to fetch in a single request at the lowest zoom.

The full dataset is 18 000 × 36 000 pixels (0.01° resolution, global).

| Levels | Coarsest shape (lat × lon) | Coarsest pixel size |
|--------|---------------------------|---------------------|
| 6      | 281 × 562                 | ~0.64°              |
| 7      | 140 × 281                 | ~1.28°              |
| 8      | 70  × 140                 | ~2.57°              |

7 levels gives a manageable coarsest tile (~140 × 281 pixels) while still preserving meaningful spatial detail at the finest level.

### Coarsening method
`method='mean'` averages LST values spatially at each coarser level. This is the correct aggregation for temperature — spatial means preserve the visual signal better than min/max.

### Non-spatial dimension
The `year` dimension is preserved at every pyramid level — zarr-layer handles it as a non-spatial dimension that the user can slice interactively.

In [5]:
# For avg daytime LST, we should use the rainbow cmap with default vmin -45C, and default vmax 45C. For max daytime LST, cmap should be fire and default vmin should be -35C and default vmax should be 55C.
layer_hints = {
    'avg_daytime_lst': ZarrLayerVarConfig(clim=[-45, 45], colormap='rainbow'),
    'max_daytime_lst': ZarrLayerVarConfig(clim=[-35, 55], colormap='fire'),
}

In [6]:
N_LEVELS = 7

pyramid = create_pyramid(
    ds_crs,
    levels=N_LEVELS,
    x_dim='longitude',
    y_dim='latitude',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # 500 KB chunks — web-friendly
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

pyramid

Pyramid(datatree=<xarray.DataTree 'root'>
Group: /
│   Attributes:
│       zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
│       multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
│       proj:code:           GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6...
│       spatial:dimensions:  ['latitude', 'longitude']
│       spatial:transform:   [0.009999999999990905, 0.0, -180.0, 0.0, -0.01000000...
│       spatial:bbox:        [-180.0, -90.00000000009209, 179.99999999967258, 90.0]
│       spatial:shape:       [18000, 36000]
│       zarr-layer:          {'avg_daytime_lst': {'clim': [-45, 45], 'colormap': ...
├── Group: /0
│       Dimensions:          (year: 3, latitude: 18000, longitude: 36000)
│       Coordinates:
│         * year             (year) int64 24B 2020 2021 2022
│         * latitude         (latitude) float64 144kB 90.0 89.98 89.97 ... -89.99 -90.0
│         * longitude        (longitude) float64 288kB -180.0 -180.

In [7]:
for level in pyramid.dt:
    print(f"Level: {level}")
    print(pyramid.dt[level].ds.dims)
    print(f"  avg_daytime_lst encoding: {pyramid.dt[level].ds['avg_daytime_lst'].encoding}")
    print(f"  max_daytime_lst encoding: {pyramid.dt[level].ds['max_daytime_lst'].encoding}")

Level: 0
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 18000, 'longitude': 36000})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 1
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 9000, 'longitude': 18000})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 2
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 4500, 'longitude': 9000})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 3
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 2250, 'longitude': 4500})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 4
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 1125, 'longitude': 2250})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 5
FrozenMappingWarningOnValuesAccess({'year': 3, 'latitude': 562, 'longitude': 1125})
  avg_daytime_lst encoding: {}
  max_daytime_lst encoding: {}
Level: 6
FrozenMappingWarningOnValuesAccess({'year': 3, 'latit

In [ ]:
# Store decoded LST (degrees C floats) as float32 — NaN propagates naturally for masked pixels.
FLOAT32_ENCODING = {
    'dtype': 'float32',
    'write_empty_chunks': False,
}

# Merge with topozarr's chunk/shard encoding
for level_path, level_enc in pyramid.encoding.items():
    for var_name in level_enc:
        level_enc[var_name].update(FLOAT32_ENCODING)

## 4. Write Pyramid to Plain Zarr v3

Write to Azure Blob Storage as plain Zarr v3 (no Icechunk). The pyramid is a derived, read-only product — versioning adds no value here, and using a plain store eliminates the Icechunk manifest lookup on every chunk fetch, halving HTTP round-trips for the web map.

The DataTree hierarchy maps directly to the Zarr group hierarchy:
```
modis_LST_multiscale_v1/
├── zarr.json        ← multiscales + layer-hints metadata
├── 0/               ← finest level (18000 × 36000)
├── ...
└── 6/               ← coarsest level (~281 × 562)
```

In [9]:
# Place the multiscale store alongside the main store, at the same container level
MULTISCALE_PREFIX ='icechunk_github_actions_demo/modis_LST_multiscale_v1'

In [ ]:
remove_existing_store = True  # set to True to delete existing store and start fresh
if remove_existing_store:

    fs = adlfs.AzureBlobFileSystem(
        account_name=config.AZURE_STORAGE_ACCOUNT,
        sas_token=config.AZURE_STORAGE_SAS_TOKEN,
    )
    prefix_path = f"{config.AZURE_CONTAINER}/{MULTISCALE_PREFIX}"
    if fs.exists(prefix_path):
        fs.rm(prefix_path, recursive=True)
        print(f"Deleted existing store at {prefix_path}")
    else:
        print(f"No existing store found at {prefix_path}, nothing to delete")

Deleted existing store at snowmelt/icechunk_github_actions_demo/modis_LST_multiscale_v1


In [12]:
%%time
fs = adlfs.AzureBlobFileSystem(
    account_name=config.AZURE_STORAGE_ACCOUNT,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
)

store = fs.get_mapper(f"{config.AZURE_CONTAINER}/{MULTISCALE_PREFIX}")

pyramid.dt.to_zarr(store, mode='w', encoding=pyramid.encoding, zarr_format=3, consolidated=False)

AZURE_URL = (
    f'https://{config.AZURE_STORAGE_ACCOUNT}.blob.core.windows.net'
    f'/{config.AZURE_CONTAINER}/{MULTISCALE_PREFIX}'
)
print(f'Written to: {AZURE_URL}')

Written to: https://uwcryo.blob.core.windows.net/snowmelt/icechunk_github_actions_demo/modis_LST_multiscale_v1
CPU times: user 7min 7s, sys: 1min 13s, total: 8min 20s
Wall time: 12min 36s


### The following code will set a default Cache-Control header on the blob container so every blob is served with caching

In [ ]:
from azure.storage.blob import BlobServiceClient, ContentSettings

blob_service = BlobServiceClient(
    account_url=f'https://{config.AZURE_STORAGE_ACCOUNT}.blob.core.windows.net',
    credential=config.AZURE_STORAGE_SAS_TOKEN,
)
container_client = blob_service.get_container_client(config.AZURE_CONTAINER)

prefix = MULTISCALE_PREFIX + '/'
blobs = list(container_client.list_blobs(name_starts_with=prefix))
blobs

In [13]:
for i, blob in enumerate(blobs):
    print(f'Processing blob {i+1}/{len(blobs)}: {blob.name}')
    container_client.get_blob_client(blob.name).set_http_headers(
        ContentSettings(cache_control='public, max-age=31536000')
    )
print(f'Set Cache-Control: public, max-age=31536000 on {len(blobs)} blobs')

Set Cache-Control: public, max-age=31536000 on 3630 blobs
